# So sánh Passive Aggressive Regressor (PAR) và PAR + Ridge

Notebook này:
1. Tự triển khai **PAR** và **PAR + Ridge** bằng NumPy (+ Numba)
2. Thử nghiệm nhiều tham số / số vòng lặp (epochs)
3. So sánh hai thuật toán với setup tốt nhất (biểu đồ Objective vs Iterations / Time)
4. Đối chiếu với chế độ mặc định của **scikit-learn** (hàm mục tiêu + thời gian)

## A. Setup & dữ liệu

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from numba import njit
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.linear_model import PassiveAggressiveRegressor, SGDRegressor

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.grid": True,
    "font.size": 12,
})

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
instagram_data = pd.read_csv("datasets/instagram_new_data.csv", encoding="utf-8-sig")

feature_cols = ["Likes", "Saves", "Comments", "Shares", "Profile Visits", "Follows"]
X = np.array(instagram_data[feature_cols], dtype=np.float64)
y = np.array(instagram_data["Impressions"], dtype=np.float64)

xtrain, xtest, ytrain, ytest = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

scaler = StandardScaler()
xtrain = scaler.fit_transform(xtrain).astype(np.float64)
xtest = scaler.transform(xtest).astype(np.float64)
ytrain = ytrain.astype(np.float64)
ytest = ytest.astype(np.float64)

print(f"Train: {xtrain.shape[0]} mẫu | Test: {xtest.shape[0]} mẫu | Features: {xtrain.shape[1]}")

## B. Hàm mục tiêu & Custom Passive Aggressive Regressor (PAR)

Hàm mất mát epsilon-insensitive trên một mẫu:

$$L_t = \max(0, |y_t - w^T x_t - b| - \epsilon)$$

Hàm mục tiêu (objective) theo dõi sau mỗi epoch:

$$J(w) = \frac{1}{n}\sum_{i=1}^{n} \max(0, |y_i - w^T x_i - b| - \epsilon)$$

Quy tắc cập nhật PAR (khi $L_t > 0$):

$$\tau_t = \frac{L_t}{\|x_t\|^2 + \frac{1}{2C}}, \quad w \leftarrow w + \tau_t \cdot \mathrm{sgn}(e_t)\, x_t$$

In [ ]:
def epsilon_insensitive_loss(X, y, w, b, epsilon):
    """Trung bình epsilon-insensitive loss trên toàn bộ tập."""
    residual = np.abs(y - (X @ w + b))
    return float(np.mean(np.maximum(0.0, residual - epsilon)))


def ridge_objective(X, y, w, b, epsilon, alpha):
    """Objective = average epsilon-insensitive loss + (alpha/2)||w||^2."""
    return epsilon_insensitive_loss(X, y, w, b, epsilon) + 0.5 * alpha * float(np.dot(w, w))


def predict_linear(X, w, b):
    return X @ w + b


def evaluate_model(X, y, w, b):
    y_pred = predict_linear(X, w, b)
    return {
        "r2": float(r2_score(y, y_pred)),
        "mae": float(mean_absolute_error(y, y_pred)),
    }

In [ ]:
@njit(cache=True)
def _par_epoch(X, y, w, b, C, epsilon, indices):
    """Một epoch cập nhật PAR (Numba)."""
    for k in range(indices.shape[0]):
        i = indices[k]
        pred = b
        for j in range(w.shape[0]):
            pred += X[i, j] * w[j]
        err = y[i] - pred
        loss = abs(err) - epsilon
        if loss > 0.0:
            xnorm = 0.0
            for j in range(w.shape[0]):
                xnorm += X[i, j] * X[i, j]
            tau = loss / (xnorm + 1.0 / (2.0 * C))
            sign = 1.0 if err >= 0.0 else -1.0
            for j in range(w.shape[0]):
                w[j] = w[j] + tau * sign * X[i, j]
            b = b + tau * sign
    return w, b


def fit_par(
    X,
    y,
    C=1.0,
    epsilon=0.1,
    max_iter=200,
    shuffle=True,
    random_state=RANDOM_STATE,
    track=True,
    checkpoint_iters=None,
):
    """Custom Passive Aggressive Regressor (NumPy + Numba)."""
    n_samples, n_features = X.shape
    w = np.zeros(n_features, dtype=np.float64)
    b = 0.0

    rng = np.random.default_rng(random_state)
    objectives = []
    times = []
    checkpoints = {}
    checkpoint_set = set(checkpoint_iters or [])

    t0 = time.perf_counter()

    # Warm-up JIT
    _ = _par_epoch(X[:1], y[:1], w.copy(), b, C, epsilon, np.array([0], dtype=np.int64))

    for epoch in range(1, max_iter + 1):
        indices = rng.permutation(n_samples).astype(np.int64) if shuffle else np.arange(n_samples, dtype=np.int64)
        w, b = _par_epoch(X, y, w, b, C, epsilon, indices)

        if track:
            objectives.append(epsilon_insensitive_loss(X, y, w, b, epsilon))
            times.append(time.perf_counter() - t0)

        if epoch in checkpoint_set:
            checkpoints[epoch] = {
                "w": w.copy(),
                "b": float(b),
                "objective": epsilon_insensitive_loss(X, y, w, b, epsilon),
                "elapsed": time.perf_counter() - t0,
            }

    elapsed = time.perf_counter() - t0
    return {
        "w": w,
        "b": float(b),
        "objectives": np.array(objectives, dtype=np.float64),
        "times": np.array(times, dtype=np.float64),
        "elapsed": elapsed,
        "final_objective": epsilon_insensitive_loss(X, y, w, b, epsilon),
        "checkpoints": checkpoints,
    }

## C. Custom PAR + Ridge Regularization

Bài toán cập nhật tại mỗi mẫu:

$$w_{t+1} = \arg\min_w \frac{1}{2}\|w - w_t\|^2 + C \cdot L_t + \frac{\alpha}{2}\|w\|^2$$

Khi $L_t > 0$, dùng cập nhật ổn định (tránh hệ số shrink âm):

$$\tau_t = \frac{L_t}{\|x_t\|^2/(1+\alpha) + \frac{1}{2C}}, \quad w \leftarrow \frac{w + \tau_t \cdot \mathrm{sgn}(e_t)\, x_t}{1+\alpha}$$

Hàm mục tiêu theo dõi:

$$J_{\mathrm{ridge}}(w) = \frac{1}{n}\sum_{i=1}^{n} \max(0, |y_i - w^T x_i - b| - \epsilon) + \frac{\alpha}{2}\|w\|^2$$

In [ ]:
@njit(cache=True)
def _par_ridge_epoch(X, y, w, b, C, epsilon, alpha, indices):
    """Một epoch PAR + Ridge (Numba), cập nhật ổn định: w <- (w + τ sgn x)/(1+α)."""
    for k in range(indices.shape[0]):
        i = indices[k]
        pred = b
        for j in range(w.shape[0]):
            pred += X[i, j] * w[j]
        err = y[i] - pred
        loss = abs(err) - epsilon
        if loss > 0.0:
            xnorm = 0.0
            for j in range(w.shape[0]):
                xnorm += X[i, j] * X[i, j]
            # Nghiệm xấp xỉ của: 1/2||w-w_t||^2 + C L + (α/2)||w||^2
            tau = loss / (xnorm / (1.0 + alpha) + 1.0 / (2.0 * C))
            sign = 1.0 if err >= 0.0 else -1.0
            inv = 1.0 / (1.0 + alpha)
            for j in range(w.shape[0]):
                w[j] = (w[j] + tau * sign * X[i, j]) * inv
            b = b + tau * sign
    return w, b


def fit_par_ridge(
    X,
    y,
    C=1.0,
    epsilon=0.1,
    alpha=0.01,
    max_iter=200,
    shuffle=True,
    random_state=RANDOM_STATE,
    track=True,
    checkpoint_iters=None,
):
    """Custom Passive Aggressive Regressor + Ridge (NumPy + Numba)."""
    n_samples, n_features = X.shape
    w = np.zeros(n_features, dtype=np.float64)
    b = 0.0

    rng = np.random.default_rng(random_state)
    objectives = []
    times = []
    checkpoints = {}
    checkpoint_set = set(checkpoint_iters or [])

    t0 = time.perf_counter()

    # Warm-up JIT
    _ = _par_ridge_epoch(
        X[:1], y[:1], w.copy(), b, C, epsilon, alpha, np.array([0], dtype=np.int64)
    )

    for epoch in range(1, max_iter + 1):
        indices = rng.permutation(n_samples).astype(np.int64) if shuffle else np.arange(n_samples, dtype=np.int64)
        w, b = _par_ridge_epoch(X, y, w, b, C, epsilon, alpha, indices)

        if track:
            objectives.append(ridge_objective(X, y, w, b, epsilon, alpha))
            times.append(time.perf_counter() - t0)

        if epoch in checkpoint_set:
            checkpoints[epoch] = {
                "w": w.copy(),
                "b": float(b),
                "objective": ridge_objective(X, y, w, b, epsilon, alpha),
                "elapsed": time.perf_counter() - t0,
            }

    elapsed = time.perf_counter() - t0
    return {
        "w": w,
        "b": float(b),
        "objectives": np.array(objectives, dtype=np.float64),
        "times": np.array(times, dtype=np.float64),
        "elapsed": elapsed,
        "final_objective": ridge_objective(X, y, w, b, epsilon, alpha),
        "final_loss_only": epsilon_insensitive_loss(X, y, w, b, epsilon),
        "checkpoints": checkpoints,
    }

## D. Thử nghiệm tham số cho PAR

Thử nhiều giá trị `C` (độ mạnh cập nhật / bước), `epsilon` (vùng không nhạy), và các mốc `max_iter` (độ dài huấn luyện).

In [ ]:
C_VALUES = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
EPS_VALUES = [0.01, 0.1, 1.0]
MAX_ITER_CHECKPOINTS = [50, 100, 200, 500, 1000]
TUNE_EPOCHS = 200  # epoch cố định khi quét C / epsilon

In [ ]:
# --- D1. Quét C với epsilon cố định ---
par_C_rows = []
for C in C_VALUES:
    res = fit_par(xtrain, ytrain, C=C, epsilon=0.1, max_iter=TUNE_EPOCHS, track=True)
    metrics = evaluate_model(xtest, ytest, res["w"], res["b"])
    par_C_rows.append({
        "C": C,
        "epsilon": 0.1,
        "max_iter": TUNE_EPOCHS,
        "objective": res["final_objective"],
        "time_s": res["elapsed"],
        "r2": metrics["r2"],
        "mae": metrics["mae"],
    })
    print(
        f"PAR | C={C:<7} | J={res['final_objective']:.4f} | "
        f"time={res['elapsed']:.2f}s | R²={metrics['r2']:.4f} | MAE={metrics['mae']:.2f}"
    )

df_par_C = pd.DataFrame(par_C_rows)
df_par_C

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(df_par_C["C"], df_par_C["objective"], marker="o")
axes[0].set_xscale("log")
axes[0].set_xlabel("C")
axes[0].set_ylabel("Objective J(w)")
axes[0].set_title("PAR: Objective vs C (ε=0.1, epochs=200)")

axes[1].plot(df_par_C["C"], df_par_C["mae"], marker="o", color="tab:orange")
axes[1].set_xscale("log")
axes[1].set_xlabel("C")
axes[1].set_ylabel("MAE (test)")
axes[1].set_title("PAR: MAE vs C")

plt.tight_layout()
plt.show()

best_par_C = float(df_par_C.loc[df_par_C["objective"].idxmin(), "C"])
print(f"C tốt nhất theo objective: {best_par_C}")

In [ ]:
# --- D2. Quét epsilon với C tốt nhất ---
par_eps_rows = []
for eps in EPS_VALUES:
    res = fit_par(xtrain, ytrain, C=best_par_C, epsilon=eps, max_iter=TUNE_EPOCHS, track=True)
    metrics = evaluate_model(xtest, ytest, res["w"], res["b"])
    par_eps_rows.append({
        "C": best_par_C,
        "epsilon": eps,
        "max_iter": TUNE_EPOCHS,
        "objective": res["final_objective"],
        "time_s": res["elapsed"],
        "r2": metrics["r2"],
        "mae": metrics["mae"],
    })
    print(
        f"PAR | ε={eps:<5} | J={res['final_objective']:.4f} | "
        f"time={res['elapsed']:.2f}s | R²={metrics['r2']:.4f} | MAE={metrics['mae']:.2f}"
    )

df_par_eps = pd.DataFrame(par_eps_rows)
display(df_par_eps)

best_par_eps = float(df_par_eps.loc[df_par_eps["objective"].idxmin(), "epsilon"])
print(f"epsilon tốt nhất theo objective: {best_par_eps}")

In [ ]:
# --- D3. Ảnh hưởng của số epochs (một lần train, lấy checkpoint) ---
par_long = fit_par(
    xtrain, ytrain,
    C=best_par_C, epsilon=best_par_eps,
    max_iter=max(MAX_ITER_CHECKPOINTS),
    track=True,
    checkpoint_iters=MAX_ITER_CHECKPOINTS,
)

par_iter_rows = []
for k in MAX_ITER_CHECKPOINTS:
    ck = par_long["checkpoints"][k]
    metrics = evaluate_model(xtest, ytest, ck["w"], ck["b"])
    par_iter_rows.append({
        "max_iter": k,
        "objective": ck["objective"],
        "time_s": ck["elapsed"],
        "r2": metrics["r2"],
        "mae": metrics["mae"],
    })
    print(
        f"PAR | epochs={k:<4} | J={ck['objective']:.4f} | "
        f"time={ck['elapsed']:.2f}s | R²={metrics['r2']:.4f} | MAE={metrics['mae']:.2f}"
    )

df_par_iter = pd.DataFrame(par_iter_rows)
display(df_par_iter)

best_par_iter = int(df_par_iter.loc[df_par_iter["objective"].idxmin(), "max_iter"])
print(f"max_iter tốt nhất theo objective: {best_par_iter}")

### Nhận xét lựa chọn tham số PAR

- **C nhỏ**: cập nhật thận trọng → hội tụ chậm, objective có thể còn cao.
- **C lớn**: cập nhật mạnh → hội tụ nhanh hơn nhưng dễ dao động / overshoot.
- **epsilon**: vùng “passive”; quá lớn khiến model ít cập nhật, quá nhỏ khiến gần như luôn aggressive.
- **max_iter**: cần đủ epochs để objective ổn định; quá nhiều có thể tốn thời gian mà lợi ích nhỏ.

Setup tốt nhất của PAR lấy theo **objective thấp nhất** ở các thí nghiệm trên.

In [ ]:
BEST_PAR = {
    "C": float(best_par_C),
    "epsilon": float(best_par_eps),
    "max_iter": int(best_par_iter),
}
print("Best PAR setup:", BEST_PAR)

## E. Thử nghiệm tham số cho PAR + Ridge

In [ ]:
ALPHA_VALUES = [0.0001, 0.001, 0.01, 0.1, 1.0]

In [ ]:
# --- E1. Quét alpha (Ridge) với C / epsilon từ PAR ---
ridge_alpha_rows = []
for alpha in ALPHA_VALUES:
    res = fit_par_ridge(
        xtrain, ytrain,
        C=BEST_PAR["C"], epsilon=BEST_PAR["epsilon"], alpha=alpha,
        max_iter=TUNE_EPOCHS, track=True,
    )
    metrics = evaluate_model(xtest, ytest, res["w"], res["b"])
    ridge_alpha_rows.append({
        "C": BEST_PAR["C"],
        "epsilon": BEST_PAR["epsilon"],
        "alpha": alpha,
        "max_iter": TUNE_EPOCHS,
        "objective": res["final_objective"],
        "loss_only": res["final_loss_only"],
        "time_s": res["elapsed"],
        "r2": metrics["r2"],
        "mae": metrics["mae"],
    })
    print(
        f"PAR+Ridge | α={alpha:<7} | J={res['final_objective']:.4f} | "
        f"loss={res['final_loss_only']:.4f} | time={res['elapsed']:.2f}s | "
        f"R²={metrics['r2']:.4f} | MAE={metrics['mae']:.2f}"
    )

df_ridge_alpha = pd.DataFrame(ridge_alpha_rows)
display(df_ridge_alpha)

best_ridge_alpha_from_scan = float(df_ridge_alpha.loc[df_ridge_alpha["objective"].idxmin(), "alpha"])
print(f"alpha tốt nhất (scan 1D): {best_ridge_alpha_from_scan}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(df_ridge_alpha["alpha"], df_ridge_alpha["objective"], marker="o")
axes[0].set_xscale("log")
axes[0].set_xlabel("alpha (Ridge)")
axes[0].set_ylabel("Objective J_ridge(w)")
axes[0].set_title("PAR+Ridge: Objective vs alpha")

axes[1].plot(df_ridge_alpha["alpha"], df_ridge_alpha["mae"], marker="o", color="tab:orange")
axes[1].set_xscale("log")
axes[1].set_xlabel("alpha (Ridge)")
axes[1].set_ylabel("MAE (test)")
axes[1].set_title("PAR+Ridge: MAE vs alpha")

plt.tight_layout()
plt.show()

In [ ]:
# --- E2. Lưới C × alpha (epsilon cố định) ---
ridge_grid_rows = []
for C in C_VALUES:
    for alpha in ALPHA_VALUES:
        res = fit_par_ridge(
            xtrain, ytrain,
            C=C, epsilon=BEST_PAR["epsilon"], alpha=alpha,
            max_iter=TUNE_EPOCHS, track=False,
        )
        obj = ridge_objective(xtrain, ytrain, res["w"], res["b"], BEST_PAR["epsilon"], alpha)
        metrics = evaluate_model(xtest, ytest, res["w"], res["b"])
        ridge_grid_rows.append({
            "C": C,
            "alpha": alpha,
            "epsilon": BEST_PAR["epsilon"],
            "max_iter": TUNE_EPOCHS,
            "objective": obj,
            "time_s": res["elapsed"],
            "r2": metrics["r2"],
            "mae": metrics["mae"],
        })

df_ridge_grid = pd.DataFrame(ridge_grid_rows)
print("Top 10 setup PAR+Ridge theo objective thấp nhất:")
display(df_ridge_grid.sort_values("objective").head(10))

best_ridge_row = df_ridge_grid.loc[df_ridge_grid["objective"].idxmin()]
best_ridge_C = float(best_ridge_row["C"])
best_ridge_alpha = float(best_ridge_row["alpha"])
print(f"Best từ lưới C×α: C={best_ridge_C}, alpha={best_ridge_alpha}")

In [ ]:
# Heatmap objective theo C × alpha
pivot = df_ridge_grid.pivot(index="alpha", columns="C", values="objective")
fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(pivot.values, aspect="auto", origin="lower")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_xlabel("C")
ax.set_ylabel("alpha")
ax.set_title("PAR+Ridge: Objective heatmap (C × alpha)")
fig.colorbar(im, ax=ax, label="Objective")
plt.tight_layout()
plt.show()

In [ ]:
# --- E3. Ảnh hưởng max_iter với best C, alpha ---
ridge_long = fit_par_ridge(
    xtrain, ytrain,
    C=best_ridge_C, epsilon=BEST_PAR["epsilon"], alpha=best_ridge_alpha,
    max_iter=max(MAX_ITER_CHECKPOINTS),
    track=True,
    checkpoint_iters=MAX_ITER_CHECKPOINTS,
)

ridge_iter_rows = []
for k in MAX_ITER_CHECKPOINTS:
    ck = ridge_long["checkpoints"][k]
    metrics = evaluate_model(xtest, ytest, ck["w"], ck["b"])
    ridge_iter_rows.append({
        "max_iter": k,
        "objective": ck["objective"],
        "time_s": ck["elapsed"],
        "r2": metrics["r2"],
        "mae": metrics["mae"],
    })
    print(
        f"PAR+Ridge | epochs={k:<4} | J={ck['objective']:.4f} | "
        f"time={ck['elapsed']:.2f}s | R²={metrics['r2']:.4f} | MAE={metrics['mae']:.2f}"
    )

df_ridge_iter = pd.DataFrame(ridge_iter_rows)
display(df_ridge_iter)

best_ridge_iter = int(df_ridge_iter.loc[df_ridge_iter["objective"].idxmin(), "max_iter"])
print(f"max_iter tốt nhất (PAR+Ridge): {best_ridge_iter}")

### Nhận xét lựa chọn tham số PAR + Ridge

- **alpha nhỏ**: gần với PAR gốc, regularization yếu.
- **alpha lớn**: thu nhỏ trọng số mạnh → ổn định hơn nhưng có thể tăng bias / tăng loss dữ liệu.
- Cần cân bằng `C` (mức aggressive) và `alpha` (mức co trọng số).
- Setup tốt nhất lấy theo **objective thấp nhất** trên lưới / checkpoint.

In [ ]:
BEST_PAR_RIDGE = {
    "C": float(best_ridge_C),
    "epsilon": float(BEST_PAR["epsilon"]),
    "alpha": float(best_ridge_alpha),
    "max_iter": int(best_ridge_iter),
}
print("Best PAR+Ridge setup:", BEST_PAR_RIDGE)

## F. So sánh PAR vs PAR+Ridge (best setups)

Vẽ 2 biểu đồ kiểu paper: **Objective vs Iterations** và **Objective vs Time**.

In [ ]:
# Chạy lại với số epoch đủ dài để so sánh đường cong hội tụ
COMPARE_EPOCHS = max(BEST_PAR["max_iter"], BEST_PAR_RIDGE["max_iter"], 500)

par_best_hist = fit_par(
    xtrain, ytrain,
    C=BEST_PAR["C"],
    epsilon=BEST_PAR["epsilon"],
    max_iter=COMPARE_EPOCHS,
    track=True,
)

ridge_best_hist = fit_par_ridge(
    xtrain, ytrain,
    C=BEST_PAR_RIDGE["C"],
    epsilon=BEST_PAR_RIDGE["epsilon"],
    alpha=BEST_PAR_RIDGE["alpha"],
    max_iter=COMPARE_EPOCHS,
    track=True,
)

par_test = evaluate_model(xtest, ytest, par_best_hist["w"], par_best_hist["b"])
ridge_test = evaluate_model(xtest, ytest, ridge_best_hist["w"], ridge_best_hist["b"])

print("=== Best PAR ===")
print(f"Setup: {BEST_PAR}")
print(f"Final objective: {par_best_hist['final_objective']:.6f}")
print(f"Time: {par_best_hist['elapsed']:.3f}s | R²={par_test['r2']:.4f} | MAE={par_test['mae']:.2f}")

print("\n=== Best PAR+Ridge ===")
print(f"Setup: {BEST_PAR_RIDGE}")
print(f"Final objective: {ridge_best_hist['final_objective']:.6f}")
print(f"Time: {ridge_best_hist['elapsed']:.3f}s | R²={ridge_test['r2']:.4f} | MAE={ridge_test['mae']:.2f}")

In [ ]:
# Biểu đồ 1: Objective vs Iterations
epochs = np.arange(1, COMPARE_EPOCHS + 1)

plt.figure(figsize=(10, 6))
plt.plot(
    epochs, par_best_hist["objectives"],
    label=f"PAR (C={BEST_PAR['C']}, ε={BEST_PAR['epsilon']})",
    linewidth=2,
)
plt.plot(
    epochs, ridge_best_hist["objectives"],
    label=(
        f"PAR+Ridge (C={BEST_PAR_RIDGE['C']}, ε={BEST_PAR_RIDGE['epsilon']}, "
        f"α={BEST_PAR_RIDGE['alpha']})"
    ),
    linewidth=2,
    linestyle="--",
)
plt.xlabel("Iterations (Epochs)")
plt.ylabel("Objective Function Value")
plt.title("Comparison of PAR vs PAR+Ridge\nTraining Objective vs Iterations")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Biểu đồ 2: Objective vs Time
plt.figure(figsize=(10, 6))
plt.plot(
    par_best_hist["times"], par_best_hist["objectives"],
    label=f"PAR (C={BEST_PAR['C']}, ε={BEST_PAR['epsilon']})",
    linewidth=2,
)
plt.plot(
    ridge_best_hist["times"], ridge_best_hist["objectives"],
    label=(
        f"PAR+Ridge (C={BEST_PAR_RIDGE['C']}, ε={BEST_PAR_RIDGE['epsilon']}, "
        f"α={BEST_PAR_RIDGE['alpha']})"
    ),
    linewidth=2,
    linestyle="--",
)
plt.xlabel("Time (seconds)")
plt.ylabel("Objective Function Value")
plt.title("Comparison of PAR vs PAR+Ridge\nTraining Objective vs Time")
plt.legend()
plt.tight_layout()
plt.show()

### Kết luận so sánh PAR vs PAR+Ridge

- Thuật toán có **đường cong objective giảm nhanh hơn / thấp hơn** trên biểu đồ Iterations và Time là lựa chọn tối ưu hóa tốt hơn cho bài toán.
- PAR+Ridge thường ổn định trọng số hơn nhờ L2; nếu objective (có penalty) thấp hơn và R²/MAE không kém, nên ưu tiên PAR+Ridge.
- Nếu regularization quá mạnh làm tăng bias rõ rệt, giữ PAR gốc với C/ε đã tinh chỉnh.
- Kết luận cụ thể phụ thuộc kết quả chạy ở các cell trên (xem bảng tổng hợp phần H).

## G. So sánh custom code với scikit-learn (default)

So sánh **hàm mục tiêu của bài toán tối ưu** và **thời gian**, không chỉ độ chính xác.

- PAR sklearn: `PassiveAggressiveRegressor()` (default)
- PAR+Ridge sklearn: `SGDRegressor(loss='epsilon_insensitive', penalty='l2', learning_rate='pa1', eta0=1.0)`

In [ ]:
def sklearn_par_objective(model, X, y, epsilon):
    """Objective PAR: average epsilon-insensitive loss."""
    y_pred = model.predict(X)
    return float(np.mean(np.maximum(0.0, np.abs(y - y_pred) - epsilon)))


def sklearn_ridge_objective(model, X, y, epsilon, alpha):
    y_pred = model.predict(X)
    loss = float(np.mean(np.maximum(0.0, np.abs(y - y_pred) - epsilon)))
    w = model.coef_.ravel()
    return loss + 0.5 * alpha * float(np.dot(w, w))

In [ ]:
# --- sklearn PAR default ---
t0 = time.perf_counter()
sk_par = PassiveAggressiveRegressor(random_state=RANDOM_STATE)
sk_par.fit(xtrain, ytrain)
sk_par_time = time.perf_counter() - t0

sk_par_eps = float(getattr(sk_par, "epsilon", 0.1))
sk_par_obj = sklearn_par_objective(sk_par, xtrain, ytrain, sk_par_eps)
sk_par_test = {
    "r2": float(r2_score(ytest, sk_par.predict(xtest))),
    "mae": float(mean_absolute_error(ytest, sk_par.predict(xtest))),
}

# Custom PAR với tham số gần default sklearn: C=1.0, epsilon=0.1, max_iter=1000
custom_par_default = fit_par(
    xtrain, ytrain, C=1.0, epsilon=0.1, max_iter=1000, track=True
)
custom_par_default_test = evaluate_model(
    xtest, ytest, custom_par_default["w"], custom_par_default["b"]
)

print("=== PAR: Custom vs sklearn default ===")
print(
    f"Custom  | J={custom_par_default['final_objective']:.6f} | "
    f"time={custom_par_default['elapsed']:.3f}s | "
    f"R²={custom_par_default_test['r2']:.4f} | MAE={custom_par_default_test['mae']:.2f}"
)
print(
    f"sklearn | J={sk_par_obj:.6f} | time={sk_par_time:.3f}s | "
    f"R²={sk_par_test['r2']:.4f} | MAE={sk_par_test['mae']:.2f}"
)

In [ ]:
# --- sklearn PAR+Ridge (SGDRegressor PA + L2) ---
sk_alpha = 0.0001  # default alpha của SGDRegressor

t0 = time.perf_counter()
sk_ridge = SGDRegressor(
    loss="epsilon_insensitive",
    penalty="l2",
    alpha=sk_alpha,
    epsilon=0.1,
    learning_rate="pa1",
    eta0=1.0,
    max_iter=1000,
    tol=1e-3,
    random_state=RANDOM_STATE,
)
sk_ridge.fit(xtrain, ytrain)
sk_ridge_time = time.perf_counter() - t0

sk_ridge_obj = sklearn_ridge_objective(sk_ridge, xtrain, ytrain, 0.1, sk_alpha)
sk_ridge_test = {
    "r2": float(r2_score(ytest, sk_ridge.predict(xtest))),
    "mae": float(mean_absolute_error(ytest, sk_ridge.predict(xtest))),
}

custom_ridge_default = fit_par_ridge(
    xtrain, ytrain,
    C=1.0, epsilon=0.1, alpha=sk_alpha,
    max_iter=1000, track=True,
)
custom_ridge_default_test = evaluate_model(
    xtest, ytest, custom_ridge_default["w"], custom_ridge_default["b"]
)

print("=== PAR+Ridge: Custom vs sklearn (SGD PA1 + L2) ===")
print(
    f"Custom  | J={custom_ridge_default['final_objective']:.6f} | "
    f"time={custom_ridge_default['elapsed']:.3f}s | "
    f"R²={custom_ridge_default_test['r2']:.4f} | MAE={custom_ridge_default_test['mae']:.2f}"
)
print(
    f"sklearn | J={sk_ridge_obj:.6f} | time={sk_ridge_time:.3f}s | "
    f"R²={sk_ridge_test['r2']:.4f} | MAE={sk_ridge_test['mae']:.2f}"
)

In [ ]:
df_sklearn_compare = pd.DataFrame([
    {
        "Model": "Custom PAR (C=1, ε=0.1, epochs=1000)",
        "Objective": custom_par_default["final_objective"],
        "Time (s)": custom_par_default["elapsed"],
        "R²": custom_par_default_test["r2"],
        "MAE": custom_par_default_test["mae"],
    },
    {
        "Model": "sklearn PassiveAggressiveRegressor (default)",
        "Objective": sk_par_obj,
        "Time (s)": sk_par_time,
        "R²": sk_par_test["r2"],
        "MAE": sk_par_test["mae"],
    },
    {
        "Model": "Custom PAR+Ridge (C=1, ε=0.1, α=1e-4, epochs=1000)",
        "Objective": custom_ridge_default["final_objective"],
        "Time (s)": custom_ridge_default["elapsed"],
        "R²": custom_ridge_default_test["r2"],
        "MAE": custom_ridge_default_test["mae"],
    },
    {
        "Model": "sklearn SGDRegressor PA1+L2 (default-like)",
        "Objective": sk_ridge_obj,
        "Time (s)": sk_ridge_time,
        "R²": sk_ridge_test["r2"],
        "MAE": sk_ridge_test["mae"],
    },
])

display(df_sklearn_compare)

labels = ["Custom PAR", "sklearn PAR", "Custom PAR+Ridge", "sklearn PA+L2"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(labels, df_sklearn_compare["Objective"], color=["#4C72B0", "#55A868", "#C44E52", "#8172B2"])
axes[0].set_ylabel("Objective Function Value")
axes[0].set_title("Custom vs sklearn: Objective")
axes[0].tick_params(axis="x", rotation=20)

axes[1].bar(labels, df_sklearn_compare["Time (s)"], color=["#4C72B0", "#55A868", "#C44E52", "#8172B2"])
axes[1].set_ylabel("Time (seconds)")
axes[1].set_title("Custom vs sklearn: Runtime")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

## H. Tổng kết

### Bảng tổng hợp setup tốt nhất

In [ ]:
summary_rows = [
    {
        "Algorithm": "PAR (best)",
        "C": BEST_PAR["C"],
        "epsilon": BEST_PAR["epsilon"],
        "alpha": None,
        "max_iter": BEST_PAR["max_iter"],
        "Objective": par_best_hist["final_objective"],
        "Time (s)": par_best_hist["elapsed"],
        "R²": par_test["r2"],
        "MAE": par_test["mae"],
    },
    {
        "Algorithm": "PAR+Ridge (best)",
        "C": BEST_PAR_RIDGE["C"],
        "epsilon": BEST_PAR_RIDGE["epsilon"],
        "alpha": BEST_PAR_RIDGE["alpha"],
        "max_iter": BEST_PAR_RIDGE["max_iter"],
        "Objective": ridge_best_hist["final_objective"],
        "Time (s)": ridge_best_hist["elapsed"],
        "R²": ridge_test["r2"],
        "MAE": ridge_test["mae"],
    },
    {
        "Algorithm": "sklearn PAR default",
        "C": 1.0,
        "epsilon": sk_par_eps,
        "alpha": None,
        "max_iter": getattr(sk_par, "n_iter_", None),
        "Objective": sk_par_obj,
        "Time (s)": sk_par_time,
        "R²": sk_par_test["r2"],
        "MAE": sk_par_test["mae"],
    },
    {
        "Algorithm": "sklearn PA+L2 default-like",
        "C": 1.0,
        "epsilon": 0.1,
        "alpha": sk_alpha,
        "max_iter": getattr(sk_ridge, "n_iter_", None),
        "Objective": sk_ridge_obj,
        "Time (s)": sk_ridge_time,
        "R²": sk_ridge_test["r2"],
        "MAE": sk_ridge_test["mae"],
    },
]

df_summary = pd.DataFrame(summary_rows)
display(df_summary)

### Kết luận chọn thuật toán / tham số

1. **Tham số quan trọng**
   - `C`: kiểm soát độ lớn bước cập nhật (aggressiveness).
   - `epsilon`: ngưỡng sai số cho phép trước khi cập nhật.
   - `alpha` (Ridge): mức co trọng số, giúp ổn định / giảm overfitting.
   - `max_iter`: cần đủ lớn để objective hội tụ; theo dõi bằng biểu đồ vs iterations/time.

2. **So sánh thuật toán**
   - Dựa vào **objective** (không chỉ R²/MAE) và tốc độ giảm theo thời gian.
   - Chọn setup best của từng nhóm rồi so sánh trực tiếp trên cùng biểu đồ.

3. **Custom vs sklearn**
   - sklearn thường nhanh hơn nhờ tối ưu C/Cython.
   - Custom giúp kiểm soát / theo dõi rõ ràng đường cong hàm mục tiêu theo từng epoch — phù hợp phân tích tối ưu hóa.
   - Objective cuối cùng nên cùng bậc nếu công thức update tương đương; lệch nhỏ có thể do chi tiết triển khai (tol, shuffle, early stopping).

4. **Gợi ý thực tế cho bài Instagram Reach**
   - Dùng setup best từ phần D/E.
   - Nếu cần ổn định trọng số trên dữ liệu nhiễu: ưu tiên **PAR+Ridge** với `alpha` vừa phải.
   - Nếu ưu tiên tốc độ triển khai production: có thể dùng sklearn với tham số đã tinh chỉnh từ thí nghiệm custom.